# Graph RAG with Email Contexts

This notebook enhances the existing Graph RAG approach by including **email_contexts** payload from graph nodes stored in Qdrant.

**Key Enhancement**: Graph nodes now include concrete email examples (prospect_email + reply) to provide better context for LLM generation.

**Note**: Uses existing graph and Qdrant data - no need to rebuild.


In [ ]:
# Imports
import json
import pandas as pd
import networkx as nx
import re

from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct, Filter, FieldCondition, MatchValue

import ollama

print("✅ Imports loaded!")


/Users/zubair/Desktop/Dev/ai-automation-agent/email-agent/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Imports loaded!


In [ ]:
# Load data
faq = pd.read_csv("../data/faq_updated.csv", encoding="utf-8")

with open("../data/generated_email_pairs.json", "r", encoding="utf-8") as f:
    labels = json.load(f)

print(f"✅ Loaded {len(faq)} FAQs")
print(f"✅ Loaded {len(labels)} labeled email pairs")


✅ Loaded 22 FAQs
✅ Loaded 104 labeled email pairs


In [ ]:
# Initialize services
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# ✅ Use Qdrant path (same as graph_rag_updated2.ipynb)
# ⚠️  IMPORTANT: Make sure backend is NOT running, or Qdrant will be locked!
# If you see "AlreadyLocked" error, stop the backend first:
#   pkill -f "backend.main:app"
#   Then restart this cell

try:
    qdrant = QdrantClient(path="qdrant_data")
    print("✅ Embedder initialized")
    print("✅ Qdrant client connected")
except RuntimeError as e:
    if "already accessed" in str(e) or "AlreadyLocked" in str(e):
        print("="*70)
        print("⚠️  ERROR: Qdrant database is locked!")
        print("="*70)
        print("The backend server is currently running and using Qdrant.")
        print()
        print("To fix this:")
        print("1. Stop the backend server:")
        print("   pkill -f 'backend.main:app'")
        print()
        print("2. Wait a few seconds")
        print()
        print("3. Restart this cell")
        print("="*70)
        raise
    else:
        raise
except Exception as e:
    print(f"❌ Error connecting to Qdrant: {e}")
    raise


✅ Embedder initialized
✅ Qdrant client connected


In [ ]:
# Load or rebuild graph from labels (same as graph_rag_updated2)
G = nx.DiGraph()

for item in labels:
    label_data = item.get("labels", {})
    topic = label_data.get("topic")
    intents = label_data.get("intents", [])
    artifacts = label_data.get("artifacts", [])
    
    if not topic and not intents and not artifacts:
        continue
    
    # Add nodes
    if topic:
        G.add_node(topic, type="topic")
    
    for intent in intents:
        G.add_node(intent, type="intent")
        if topic:
            G.add_edge(topic, intent, relation="HAS_INTENT")
    
    for artifact in artifacts:
        G.add_node(artifact, type="artifact")
        if topic:
            G.add_edge(topic, artifact, relation="USES_ARTIFACT")

print(f"✅ Graph loaded: {len(G.nodes())} nodes, {len(G.edges())} edges")


✅ Graph loaded: 23 nodes, 38 edges


In [ ]:
# Extract unique intents
unique_intents = set()
for item in labels:
    label_data = item.get("labels", {})
    intents = label_data.get("intents", [])
    unique_intents.update(intents)

unique_intents = sorted(list(unique_intents))
print(f"✅ Found {len(unique_intents)} unique intents: {unique_intents}")


✅ Found 8 unique intents: ['accept_or_decline', 'confirm', 'request_feedback', 'request_info', 'reschedule', 'schedule', 'send_materials', 'share_feedback']


In [ ]:
# Intent classification function (same as graph_rag_updated2)
def classify_multi_intent(email_text: str, available_intents: list) -> list:
    """Classify multiple intents in an email using LLM."""
    intents_str = ", ".join(available_intents)
    
    prompt = f"""You are an email intent classifier.

Available intents (choose from these EXACT labels):
{intents_str}

Email to classify:
\"\"\"{email_text}\"\"\"

Instructions:
1. Identify ALL relevant intents from the list above
2. An email can have multiple intents
3. Return ONLY a JSON array with exact labels

Return ONLY valid JSON array, nothing else:
"""
    
    try:
        response = ollama.chat(
            model="llama3",
            messages=[{"role": "user", "content": prompt}]
        )
        
        content = response["message"]["content"].strip()
        content = re.sub(r'```json\s*', '', content)
        content = re.sub(r'```\s*', '', content)
        content = content.strip()
        
        try:
            intents = json.loads(content)
            if isinstance(intents, list):
                valid_intents = [i for i in intents if i in available_intents]
                if valid_intents:
                    return valid_intents
        except json.JSONDecodeError:
            match = re.search(r'\[.*?\]', content)
            if match:
                try:
                    intents = json.loads(match.group())
                    if isinstance(intents, list):
                        valid_intents = [i for i in intents if i in available_intents]
                        if valid_intents:
                            return valid_intents
                except:
                    pass
        
        # Fallback
        for intent in available_intents:
            if intent.lower() in content.lower():
                return [intent]
        
        return ["general_inquiry"]
    except Exception as e:
        print(f"⚠️ Error: {e}")
        return ["general_inquiry"]

print("✅ classify_multi_intent loaded!")


✅ classify_multi_intent loaded!


In [ ]:
# ✅ ENHANCED: Get graph nodes WITH email contexts from Qdrant
def get_nodes_by_intents_with_contexts(intents: list, limit: int = 5) -> list:
    """
    Retrieve graph nodes related to intents AND fetch email_contexts from Qdrant.
    This is the key enhancement - includes concrete email examples!
    """
    def get_relationships(node_name: str):
        """Extract relationship information from edges."""
        if node_name not in G:
            return {"outgoing": [], "incoming": []}
        
        outgoing = []
        for successor in G.successors(node_name):
            edge_data = G.get_edge_data(node_name, successor, {})
            outgoing.append({
                "node": successor,
                "relation": edge_data.get("relation", "CONNECTED")
            })
        
        incoming = []
        for predecessor in G.predecessors(node_name):
            edge_data = G.get_edge_data(predecessor, node_name, {})
            incoming.append({
                "node": predecessor,
                "relation": edge_data.get("relation", "CONNECTED")
            })
        
        return {"outgoing": outgoing, "incoming": incoming}
    
    def get_email_contexts_from_qdrant(node_name: str):
        """
        ✅ NEW: Fetch email_contexts from Qdrant for this node.
        Searches knowledge_space collection for graph_node with matching node_name.
        """
        try:
            # Method 1: Try scroll API
            try:
                results = qdrant.scroll(
                    collection_name="knowledge_space",
                    scroll_filter=Filter(
                        must=[
                            FieldCondition(key="type", match=MatchValue(value="graph_node")),
                            FieldCondition(key="node_name", match=MatchValue(value=node_name))
                        ]
                    ),
                    limit=1
                )
                points, _ = results  # results is (points, next_page_offset)
                if points:
                    point = points[0]
                    payload = point.payload if hasattr(point, 'payload') else point
                    email_contexts = payload.get("email_contexts", [])
                    return email_contexts[:3]  # Return up to 3 examples
            except:
                # Method 2: Fallback - search all graph nodes and filter
                all_graph_nodes = qdrant.scroll(
                    collection_name="knowledge_space",
                    scroll_filter=Filter(
                        must=[FieldCondition(key="type", match=MatchValue(value="graph_node"))]
                    ),
                    limit=100  # Get all graph nodes
                )
                points, _ = all_graph_nodes
                for point in points:
                    payload = point.payload if hasattr(point, 'payload') else point
                    if payload.get("node_name") == node_name:
                        email_contexts = payload.get("email_contexts", [])
                        return email_contexts[:3]
        except Exception as e:
            print(f"  ⚠️ Could not fetch email contexts for {node_name}: {e}")
        
        return []
    
    nodes = []
    seen_names = set()
    
    for intent in intents:
        if intent in G and intent not in seen_names:
            node_data = G.nodes[intent]
            neighbors = list(G.successors(intent)) + list(G.predecessors(intent))
            relationships = get_relationships(intent)
            email_contexts = get_email_contexts_from_qdrant(intent)  # ✅ Fetch contexts!
            
            nodes.append({
                "name": intent,
                "type": node_data.get("type", "unknown"),
                "neighbors": neighbors,
                "relationships": relationships,
                "email_contexts": email_contexts  # ✅ Include email contexts!
            })
            seen_names.add(intent)
            
            # Also get connected nodes
            for neighbor in neighbors:
                if neighbor not in seen_names and len(nodes) < limit:
                    neighbor_data = G.nodes[neighbor]
                    neighbor_neighbors = list(G.successors(neighbor)) + list(G.predecessors(neighbor))
                    neighbor_relationships = get_relationships(neighbor)
                    neighbor_email_contexts = get_email_contexts_from_qdrant(neighbor)  # ✅ Fetch contexts!
                    
                    nodes.append({
                        "name": neighbor,
                        "type": neighbor_data.get("type", "unknown"),
                        "neighbors": neighbor_neighbors,
                        "relationships": neighbor_relationships,
                        "email_contexts": neighbor_email_contexts  # ✅ Include email contexts!
                    })
                    seen_names.add(neighbor)
    
    return nodes[:limit]

print("✅ get_nodes_by_intents_with_contexts loaded!")


✅ get_nodes_by_intents_with_contexts loaded!


In [ ]:
# ✅ ENHANCED: Build prompt WITH email contexts in graph section
def build_prompt_with_email_contexts(email_text, intent, faq_hits, graph_hits, expanded_graph_info, style_examples=None):
    """
    Build prompt with email contexts included in graph section.
    This is the key enhancement - graph nodes now show concrete examples!
    """
    # FAQ section (unchanged)
    faq_section = "\n".join([
        f"{i+1}. Q: {f['question']}\n   A: {f['answer']}"
        for i, f in enumerate(faq_hits)
    ]) or "None"

    # ✅ ENHANCED: Graph section WITH email contexts
    graph_lines = []
    for i, g in enumerate(graph_hits):
        node_name = g.get('node_name', '')
        node_type = g.get('node_type', 'unknown')
        neighbors = g.get('neighbors', [])
        email_contexts = g.get('email_contexts', [])  # ✅ Get email contexts!
        
        # Base node info
        line = f"{i+1}. Node: {node_name} (type={node_type}), neighbors={neighbors}"
        
        # ✅ Add email contexts if available
        if email_contexts:
            line += "\n   Email Examples:"
            for j, ctx in enumerate(email_contexts[:2], 1):  # Show up to 2 examples
                subject = ctx.get('subject', 'No Subject')
                prospect_email = ctx.get('prospect_email', '')[:150]
                reply = ctx.get('reply', '')[:150]
                
                if prospect_email:
                    line += f"\n     Example {j} - Subject: {subject}"
                    line += f"\n     Email: {prospect_email}..."
                if reply:
                    line += f"\n     Reply: {reply}..."
        
        graph_lines.append(line)
    
    graph_section = "\n".join(graph_lines) if graph_lines else "None"

    # Expansion section (unchanged)
    expansion_section = "\n".join([
        f"{i+1}. {node} → {neighbors}"
        for i, (node, neighbors) in enumerate(expanded_graph_info.items())
    ]) or "None"

    # Style section (unchanged)
    style_section = ""
    if style_examples:
        style_section = "\n".join([
            f"{i+1}. Style Example {i+1} (Zubair's writing tone):\n   \"{s['reply_chunk'][:250]}...\""
            for i, s in enumerate(style_examples[:3])
        ])
    else:
        style_section = "None"

    prompt = f"""
You are **Zubair**, a graduate student known for being polite, proactive, and clear in communication.

Your job is to draft a short, natural, and professional email reply.

Keep it warm but not overly formal — think of how a thoughtful student would respond to a professor, coordinator, or peer.

**IMPORTANT**: Match the writing style shown in the style examples below. These are examples of Zubair's actual writing tone. Use similar phrasing, level of formality, and structure.

---

✉️ **Incoming Email**
\"\"\"{email_text}\"\"\"

🎯 **Detected Intent**: {intent}

📘 **Relevant FAQs** (Content Context)
{faq_section}

🧩 **Graph Context** (Structured Relationships + Email Examples)
{graph_section}

🔗 **Related Concepts**
{expansion_section}

✍️ **Writing Style Examples** (Style Anchor - Match This Tone)
{style_section}

---

Write your reply:
- Match the tone and style from the examples above
- Use similar phrasing, level of formality, and structure
- Acknowledge the sender and context
- If an action is requested, confirm or ask a polite follow-up question
- Keep the reply under 120 words
- Do NOT invent facts — only use what's in context
"""
    return prompt

print("✅ build_prompt_with_email_contexts loaded!")


✅ build_prompt_with_email_contexts loaded!


In [ ]:
# ✅ ENHANCED: Answer email function WITH email contexts
def answer_email_with_contexts(email_text: str, top_k: int = 6, show_context: bool = True):
    """
    Enhanced email answering WITH email contexts from graph nodes.
    """
    
    # Step 1: Multi-intent classification
    intents = classify_multi_intent(email_text, list(unique_intents))
    primary_intent = intents[0] if intents else "general_inquiry"
    
    if show_context:
        print("="*70)
        print("🎯 INTENT CLASSIFICATION")
        print("="*70)
        print(f"Detected Intents: {intents}")
        print(f"Primary Intent: {primary_intent}\n")
    
    # Step 2: Embed query
    q_vec = embedder.encode([email_text])[0].tolist()

    # Step 3: Search FAQs
    try:
        faq_search_results = qdrant.query_points(
            collection_name="knowledge_space",
            query=q_vec,
            limit=top_k,
            query_filter=Filter(
                must=[FieldCondition(key="type", match=MatchValue(value="faq"))]
            )
        ).points
    except:
        all_hits = qdrant.search(
            collection_name="knowledge_space",
            query_vector=q_vec,
            limit=top_k * 2,
            with_payload=True
        )
        faq_search_results = [h for h in all_hits if h.payload.get("type") == "faq"][:top_k]
    
    faq_hits = []
    for h in faq_search_results:
        p = h.payload if hasattr(h, 'payload') else h
        if p.get("type") == "faq":
            score = h.score if hasattr(h, 'score') else 0.0
            faq_hits.append({"score": score, **p})

    if show_context:
        print("="*70)
        print("📚 FAQ RETRIEVAL")
        print("="*70)
        print(f"Retrieved {len(faq_hits)} FAQ hits\n")
        for i, f in enumerate(faq_hits[:3], 1):
            print(f"  {i}. [Score {f['score']:.3f}] Q: {f['question']}")
            print(f"     A: {f['answer'][:80]}...\n")

    # Step 4: ✅ ENHANCED - Get graph nodes WITH email contexts
    intent_graph_nodes = get_nodes_by_intents_with_contexts(intents, limit=5)
    
    graph_hits = []
    for node in intent_graph_nodes:
        graph_hits.append({
            "score": 0.75,
            "node_name": node["name"],
            "node_type": node["type"],
            "neighbors": node["neighbors"],
            "relationships": node.get("relationships", {"outgoing": [], "incoming": []}),
            "email_contexts": node.get("email_contexts", [])  # ✅ Include contexts!
        })
    
    if show_context:
        print("="*70)
        print("🕸️  GRAPH RETRIEVAL (WITH EMAIL CONTEXTS)")
        print("="*70)
        print(f"Retrieved {len(graph_hits)} graph nodes\n")
        for i, g in enumerate(graph_hits, 1):
            print(f"  {i}. Node: {g['node_name']} (type: {g['node_type']})")
            print(f"     Neighbors: {', '.join(g.get('neighbors', [])[:5])}")
            email_contexts = g.get('email_contexts', [])
            if email_contexts:
                print(f"     ✅ Email Contexts: {len(email_contexts)} examples")
                for j, ctx in enumerate(email_contexts[:2], 1):
                    print(f"        Example {j}: Subject=\"{ctx.get('subject', 'N/A')[:50]}\"")
            else:
                print(f"     ⚠️  No email contexts found")
            print()

    # Step 5: Graph expansion
    expanded_graph_info = {}
    for g in graph_hits:
        node = g["node_name"]
        if node in G:
            neighbors = list(G.successors(node)) + list(G.predecessors(node))
            expanded_graph_info[node] = neighbors

    # Step 6: Style retrieval
    try:
        style_hits = qdrant.search(
            collection_name="writing_style",
            query_vector=q_vec,
            limit=3,
            with_payload=True
        )
        
        style_examples = []
        for hit in style_hits:
            style_examples.append({
                "score": hit.score,
                "reply_chunk": hit.payload.get("reply_chunk", ""),
                "subject": hit.payload.get("subject", ""),
                "intent": hit.payload.get("intent", "")
            })
    except Exception as e:
        if show_context:
            print(f"⚠️ Style search error: {e}")
        style_examples = []

    if show_context:
        print("="*70)
        print("✍️  STYLE RETRIEVAL")
        print("="*70)
        print(f"Retrieved {len(style_examples)} style examples\n")
        for i, s in enumerate(style_examples, 1):
            print(f"  {i}. [Score {s['score']:.3f}] {s['reply_chunk'][:80]}...\n")

    # Step 7: ✅ Build enhanced prompt WITH email contexts
    prompt = build_prompt_with_email_contexts(
        email_text, 
        ", ".join(intents),
        faq_hits, 
        graph_hits, 
        expanded_graph_info,
        style_examples
    )

    if show_context:
        print("="*70)
        print("📝 PROMPT PREVIEW (Graph Section)")
        print("="*70)
        # Extract and show graph section
        graph_start = prompt.find("🧩 **Graph Context**")
        graph_end = prompt.find("🔗 **Related Concepts**")
        if graph_start != -1 and graph_end != -1:
            graph_section_preview = prompt[graph_start:graph_end]
            print(graph_section_preview[:1000] + "..." if len(graph_section_preview) > 1000 else graph_section_preview)
        print()

    # Step 8: Generate reply
    resp = ollama.chat(
        model="llama3",
        messages=[{"role": "user", "content": prompt}]
    )
    reply_text = resp["message"]["content"]

    # Step 9: Confidence scoring
    top_score = faq_hits[0]["score"] if faq_hits else 0.0
    auto_send = top_score >= 0.85

    return {
        "reply": reply_text,
        "intents": intents,
        "top_score": top_score,
        "auto_send": auto_send,
        "faq_hits": faq_hits,
        "graph_hits": graph_hits,
        "style_examples": style_examples
    }

print("✅ answer_email_with_contexts loaded!")


✅ answer_email_with_contexts loaded!


## Testing with Examples from graph_rag_updated2

Testing the enhanced approach with email contexts on real examples.


In [ ]:
# Test Example 1: Schedule + Send Materials
test_email_1 = """Hi Zubair,
Thank you for reaching out. To proceed with your interest in the Advanced Analytics position at Google, kindly share your resume and provide the right time to connect with you for an online meeting.
Regards,
Arjun Das
Talent Acquisition
Google Inc."""

print("="*70)
print("🧪 TEST EXAMPLE 1: Schedule + Send Materials")
print("="*70)
print(f"\n📧 Input Email:\n{test_email_1}\n")

result = answer_email_with_contexts(test_email_1, show_context=True)

print("\n" + "="*70)
print("📧 FINAL RESULT")
print("="*70)
print(f"Intents: {result['intents']}")
print(f"Confidence: {result['top_score']:.3f}")
print(f"Auto-send: {result['auto_send']}")
print(f"\n✍️ Generated Reply:")
print("-"*70)
print(result['reply'])
print("-"*70)


🧪 TEST EXAMPLE 1: Schedule + Send Materials

📧 Input Email:
Hi Zubair,
Thank you for reaching out. To proceed with your interest in the Advanced Analytics position at Google, kindly share your resume and provide the right time to connect with you for an online meeting.
Regards,
Arjun Das
Talent Acquisition
Google Inc.

🎯 INTENT CLASSIFICATION
Detected Intents: ['request_info', 'share_feedback', 'schedule']
Primary Intent: request_info

📚 FAQ RETRIEVAL
Retrieved 6 FAQ hits

  1. [Score 0.422] Q: Can I see Zubair's Data Science resume?
     A: Yes, here is the link to Zubair's Data Science resume: https://drive.google.com/...

  2. [Score 0.399] Q: Can I see Zubair's Software Engineering resume?
     A: Yes, here is the link to Zubair's Software Engineering resume: https://drive.goo...

  3. [Score 0.361] Q: Does Zubair accept research or job opportunities?
     A: Zubair considers opportunities on a case-by-case basis. Please reach out with de...

🕸️  GRAPH RETRIEVAL (WITH EMAIL CONTEXT

In [ ]:
# Test Example 2: Request Info
test_email_2 = "Hi Zubair, I hope you're doing well. Can I get your linkedin profile and calendly to connect?"

print("="*70)
print("🧪 TEST EXAMPLE 2: Request Info")
print("="*70)
print(f"\n📧 Input Email:\n{test_email_2}\n")

result = answer_email_with_contexts(test_email_2, show_context=True)

print("\n" + "="*70)
print("📧 FINAL RESULT")
print("="*70)
print(f"Intents: {result['intents']}")
print(f"Confidence: {result['top_score']:.3f}")
print(f"Auto-send: {result['auto_send']}")
print(f"\n✍️ Generated Reply:")
print("-"*70)
print(result['reply'])
print("-"*70)


🧪 TEST EXAMPLE 2: Request Info

📧 Input Email:
Hi Zubair, I hope you're doing well. Can I get your linkedin profile and calendly to connect?

🎯 INTENT CLASSIFICATION
Detected Intents: ['request_info']
Primary Intent: request_info

📚 FAQ RETRIEVAL
Retrieved 6 FAQ hits

  1. [Score 0.520] Q: What is Zubair's LinkedIn profile?
     A: https://www.linkedin.com/in/zubair-atha/...

  2. [Score 0.457] Q: How can I contact Zubair for a call?
     A: You can reach Zubair by phone at +16463925601 or schedule a meeting via Calendly...

  3. [Score 0.440] Q: What is Zubair's Calendly link?
     A: https://calendly.com/za2366-columbia/30min...

🕸️  GRAPH RETRIEVAL (WITH EMAIL CONTEXTS)
Retrieved 4 graph nodes

  1. Node: request_info (type: intent)
     Neighbors: Recruiter/Job Search, Group/Event Coordination, Feedback & Reviews
     ✅ Email Contexts: 3 examples
        Example 1: Subject="Opportunity at TechCorp - We'd Love to Know More A"
        Example 2: Subject="Job Opening - Data Scientist 

In [ ]:
# Test Example 3: Confirm
test_email_3 = (
    "Hi Zubair,"
    "Did you get a chance to send the updated project slides to Prof. Rivera?"
    "He emailed me this morning asking for them, so I just wanted to confirm."
    "Let me know!"
    "Best,"
    "Alex"
)

print("="*70)
print("🧪 TEST EXAMPLE 3: Confirm")
print("="*70)
print(f"\n📧 Input Email:\n{test_email_3}\n")

result = answer_email_with_contexts(test_email_3, show_context=True)

print("\n" + "="*70)
print("📧 FINAL RESULT")
print("="*70)
print(f"Intents: {result['intents']}")
print(f"Confidence: {result['top_score']:.3f}")
print(f"Auto-send: {result['auto_send']}")
print(f"\n✍️ Generated Reply:")
print("-"*70)
print(result['reply'])
print("-"*70)


In [ ]:
# Test Example 4: Send Materials
test_email_4 = (
    "Hello Zubair,"
    "We're currently on the lookout for skilled individuals in machine learning, and your profile caught our eye. Would you mind sharing your resume with us? Can you share your calendly for call."
    "Best,"
    "Salman Khan"
    "Bhai Dynamics"
)

print("="*70)
print("🧪 TEST EXAMPLE 4: Send Materials")
print("="*70)
print(f"\n📧 Input Email:\n{test_email_4}\n")

result = answer_email_with_contexts(test_email_4, show_context=True)

print("\n" + "="*70)
print("📧 FINAL RESULT")
print("="*70)
print(f"Intents: {result['intents']}")
print(f"Confidence: {result['top_score']:.3f}")
print(f"Auto-send: {result['auto_send']}")
print(f"\n✍️ Generated Reply:")
print("-"*70)
print(result['reply'])
print("-"*70)


🧪 TEST EXAMPLE 4: Send Materials

📧 Input Email:
Hello Zubair,We're currently on the lookout for skilled individuals in machine learning, and your profile caught our eye. Would you mind sharing your resume with us? Can you share your calendly for call.Best,Salman KhanBhai Dynamics

🎯 INTENT CLASSIFICATION
Detected Intents: ['request_info', 'request_feedback']
Primary Intent: request_info

📚 FAQ RETRIEVAL
Retrieved 6 FAQ hits

  1. [Score 0.462] Q: Can I see Zubair's Software Engineering resume?
     A: Yes, here is the link to Zubair's Software Engineering resume: https://drive.goo...

  2. [Score 0.461] Q: Can I see Zubair's Data Science resume?
     A: Yes, here is the link to Zubair's Data Science resume: https://drive.google.com/...

  3. [Score 0.413] Q: Does Zubair have both Data Science and Software Engineering resumes?
     A: Yes, Zubair has two separate resumes: Data Science resume (https://drive.google....

🕸️  GRAPH RETRIEVAL (WITH EMAIL CONTEXTS)
Retrieved 5 graph nodes

 